1. Setup

In [3]:
from src.bronze.pipeline import BronzePipeline

pipeline = BronzePipeline()
spark = pipeline.spark

2. CSV

In [4]:
csv_output_path = pipeline.run(
    source_path="/app/data/raw_local/RAW/2024-07-13_0800/CONTROLE DE MEDICOES E PAGAMENTOS/ControleMedicoesPagamentos.csv",
    source_type="csv",
    dataset_name="controle_medicoes_pagamentos",
    snapshot_date="2024-07-13_0800",
    source_file="ControleMedicoesPagamentos.csv",
    file_hash="manual_test_hash"
)

print(csv_output_path)
spark.read.parquet(csv_output_path).select("_snapshot_date", "_source_file", "_source_type").show(5)

s3a://contracts/bronze/source_type=csv/dataset=controle_medicoes_pagamentos/snapshot_date=2024-07-13_0800/
+---------------+--------------------+------------+
| _snapshot_date|        _source_file|_source_type|
+---------------+--------------------+------------+
|2024-07-13_0800|ControleMedicoesP...|         csv|
|2024-07-13_0800|ControleMedicoesP...|         csv|
|2024-07-13_0800|ControleMedicoesP...|         csv|
|2024-07-13_0800|ControleMedicoesP...|         csv|
|2024-07-13_0800|ControleMedicoesP...|         csv|
+---------------+--------------------+------------+
only showing top 5 rows



3. XLSX

In [5]:
xlsx_output_path = pipeline.run(
    source_path="/app/data/raw_local/RAW/2024-07-13_0800/CONTROLE DE MEDICOES EM ANDAMENTO/Exportação_bm_acompanhamento.xlsx",
    source_type="xlsx",
    dataset_name="controle_medicoes_andamento",
    snapshot_date="2024-07-13_0800",
    source_file="Exportação_bm_acompanhamento.xlsx",
    file_hash="manual_test_hash"
)

print(xlsx_output_path)
spark.read.parquet(xlsx_output_path).select("_snapshot_date", "_source_file", "_source_type").show(5)

s3a://contracts/bronze/source_type=xlsx/dataset=controle_medicoes_andamento/snapshot_date=2024-07-13_0800/
+---------------+--------------------+------------+
| _snapshot_date|        _source_file|_source_type|
+---------------+--------------------+------------+
|2024-07-13_0800|Exportação_bm_aco...|        xlsx|
|2024-07-13_0800|Exportação_bm_aco...|        xlsx|
|2024-07-13_0800|Exportação_bm_aco...|        xlsx|
|2024-07-13_0800|Exportação_bm_aco...|        xlsx|
|2024-07-13_0800|Exportação_bm_aco...|        xlsx|
+---------------+--------------------+------------+
only showing top 5 rows



4. XLSB / NACT

In [6]:
xlsb_output_path = pipeline.run(
    source_path="/app/data/raw_local/RAW/2024-07-13_0800/NACT/202211_ADMIN.xlsb",
    source_type="xlsb",
    dataset_name="nact",
    snapshot_date="2024-07-13_0800",
    source_file="202211_ADMIN.xlsb",
    file_hash="manual_test_hash"
)

print(xlsb_output_path)
spark.read.parquet(xlsb_output_path).select("_snapshot_date", "_source_file", "_source_type").show(5)

26/06/24 14:19:51 WARN TaskSetManager: Stage 7 contains a task of very large size (1043 KiB). The maximum recommended task size is 1000 KiB.


s3a://contracts/bronze/source_type=xlsb/dataset=nact/snapshot_date=2024-07-13_0800/
+---------------+-----------------+------------+
| _snapshot_date|     _source_file|_source_type|
+---------------+-----------------+------------+
|2024-07-13_0800|202211_ADMIN.xlsb|        xlsb|
|2024-07-13_0800|202211_ADMIN.xlsb|        xlsb|
|2024-07-13_0800|202211_ADMIN.xlsb|        xlsb|
|2024-07-13_0800|202211_ADMIN.xlsb|        xlsb|
|2024-07-13_0800|202211_ADMIN.xlsb|        xlsb|
+---------------+-----------------+------------+
only showing top 5 rows



5. Sumário

In [7]:
validation_results = {
    "csv": csv_output_path,
    "xlsx": xlsx_output_path,
    "xlsb": xlsb_output_path,
}

validation_results

{'csv': 's3a://contracts/bronze/source_type=csv/dataset=controle_medicoes_pagamentos/snapshot_date=2024-07-13_0800/',
 'xlsx': 's3a://contracts/bronze/source_type=xlsx/dataset=controle_medicoes_andamento/snapshot_date=2024-07-13_0800/',
 'xlsb': 's3a://contracts/bronze/source_type=xlsb/dataset=nact/snapshot_date=2024-07-13_0800/'}

5. Isso libera o worker para o próximo notebook.

In [8]:
#spark.stop()

### Batch Validation

In [9]:
# 1. Run Bronze Batch
from src.bronze.batch import run_bronze_batch

bronze_batch_df = run_bronze_batch()

bronze_batch_df["status"].value_counts()

status
SUCCESS    1
Name: count, dtype: int64

In [10]:
# 2. Execution Summary
bronze_batch_df[
    [
        "execution_id",
        "source_file",
        "dataset_name",
        "source_type",
        "snapshot_date",
        "status",
        "duration_seconds",
    ]
].head(10)

,execution_id,source_file,dataset_name,source_type,snapshot_date,status,duration_seconds
0,bronze_20260624_141953_47c60359,ControleMedicoesPagamentos.csv,controle_medicoes_pagamentos,csv,2024-07-13_0800,SUCCESS,0.893502


In [11]:
# 3. Failed Records
bronze_batch_df[
    bronze_batch_df["status"] == "FAILED"
][
    [
        "source_file",
        "snapshot_date",
        "dataset_name",
        "error_message",
    ]
]

,source_file,snapshot_date,dataset_name,error_message


In [12]:
# Validar o log persistido:
from src.config.settings import Settings
import pandas as pd

pd.read_csv(Settings.BRONZE_EXECUTION_LOG_PATH).head(10)

,execution_id,source_file,dataset_name,status,start_time,end_time,duration_seconds,error_message,snapshot_date,source_type,file_hash,bronze_path
0,bronze_20260624_141953_47c60359,ControleMedicoesPagamentos.csv,controle_medicoes_pagamentos,SUCCESS,2026-06-24T14:19:53.161234+00:00,2026-06-24T14:19:54.054736+00:00,0.893502,NaN,2024-07-13_0800,csv,b73d09f9dc6766eb918f3e1149649f758e0338d0f42f86...,s3a://contracts/bronze/source_type=csv/dataset...


### Idempotência
Ex: arquivo processado 10 vezes

Em produção isso é inaceitável.
se source_file + snapshot_date + file_hash já foi processado com SUCCESS
→ não reprocessar


In [13]:
from src.bronze.batch import run_bronze_batch

bronze_batch_df = run_bronze_batch(force_reprocess=False)

bronze_batch_df["status"].value_counts()

status
SKIPPED    1
SUCCESS    1
Name: count, dtype: int64